# 02 — LSTNet adaptado: pH da estação EF01 (CETESB)

**Objetivo:** testar se um recorrente com viés sazonal em **resolução nativa de 5 min** bate o sazonal-naive (MAE 0,0501 rolante / 0,0466 holdout diário), no mesmo desenho do 00/01 (L=8640/H=288 p/ avaliação, split 70/15/15 + holdout de 10 dias, 10 origens diárias).
**Arquitetura (Lai et al. 2018, adaptada univariada):** Conv1D (k=12, stride 6) sobre `LN=2016` passos nativos (7 dias) + canais hora-do-dia → GRU + recurrent-skip `p=48` (mesma fase do dia anterior) + **atalho AR-288 linear em paralelo** (escala/nível) + RevIN por janela. Saída **direta H=288, sem expansão**.
**Dados:** `dados/ef01-mogi-das-cruzes_ph_2026-06-01_a_2026-08-31.csv` — ver `dados/README.md`.

In [1]:
import json
import random
import time
import warnings
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error
from statsmodels.tsa.seasonal import STL
from statsmodels.tsa.stattools import adfuller

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

warnings.filterwarnings("ignore")
plt.rcParams.update({"figure.dpi": 110})

ROOT = Path.cwd() if (Path.cwd() / "dados").exists() else Path.cwd().parent
CSV = ROOT / "dados" / "ef01-mogi-das-cruzes_ph_2026-06-01_a_2026-08-31.csv"
OUT = ROOT / "resultados" / "02-lstnet-ph"
(OUT / "modelos").mkdir(parents=True, exist_ok=True)
(OUT / "figs").mkdir(parents=True, exist_ok=True)

# --- protocolo travado (igual ao 00) ---
L, H = 8640, 288
SEASON = 288
INTERP_LIMIT = 24
PROVISORIO_CORTE = "2026-08-22 09:00"
HOLDOUT_DIAS = 10
# --- LSTNet nativo 5 min (Lai et al. 2018, adaptado univariado) ---
LN, HN = 2016, 288        # contexto nativo (7 dias); alvo direto = H (sem expansão)
CONV_CH, CONV_K, CONV_S = 32, 12, 6   # conv1d: 2016 -> 335 passos
GRU_H, SKIP_H, SKIP_P = 64, 32, 48    # skip p=48 pós-conv = 288 passos de 5 min (1 dia)
AR_Q = 288                # atalho AR linear sobre as últimas 288 obs
BATCH, LR = 256, 1e-3
MAX_EPOCHS, PATIENCE = 60, 10
TRAIN_STRIDE, VAL_STRIDE = 4, 4
DROPOUT = 0.1
SEED = 42


random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = torch.device("cpu")
print("ROOT:", ROOT, "| CSV existe:", CSV.exists(), "| torch:", torch.__version__)


ROOT: /home/marcos/Projetos/temporal-model | CSV existe: True | torch: 2.14.0+cpu


## 1. Carga
Formato CETESB: `;`, decimal com vírgula, `windows-1252`, linha 1 = validação, linha 2 = cabeçalho.

In [2]:
df = pd.read_csv(CSV, sep=";", decimal=",", encoding="windows-1252",
                 skiprows=1, parse_dates=["Data hora"], dayfirst=True, na_values=[""])
df = df.rename(columns={"Data hora": "ds", "pH": "y"}).sort_values("ds").reset_index(drop=True)
print(df.shape, df["ds"].min(), "→", df["ds"].max())
print("faltantes:", int(df['y'].isna().sum()), f"({100*df['y'].isna().mean():.1f}%)")
df.describe()


(26209, 2) 2026-06-01 00:00:00 → 2026-08-31 00:00:00
faltantes: 4794 (18.3%)


,ds,y
count,26209,21415.000000
mean,2026-07-16 12:00:00,6.142466
min,2026-06-01 00:00:00,5.560000
25%,2026-06-23 18:00:00,5.970000
50%,2026-07-16 12:00:00,6.170000
75%,2026-08-08 06:00:00,6.280000
max,2026-08-31 00:00:00,6.590000
std,NaN,0.198893


## 2. EDA — perfil, faltantes e ciclo diário

In [3]:
isna = df["y"].isna().to_numpy()
gaps = np.diff(np.concatenate([[0], np.where(~isna)[0], [len(isna)]])) - 1
print(f"maior gap: {gaps.max()} passos = {gaps.max()*5/60:.1f} h | gaps > 24 passos: {(gaps > 24).sum()}")

fig, ax = plt.subplots(3, 1, figsize=(12, 9), sharex=False)
ax[0].plot(df["ds"], df["y"], lw=0.4)
ax[0].axvline(pd.Timestamp(PROVISORIO_CORTE), color="r", ls="--", lw=1)
ax[0].set_title("pH EF01 — série completa (vermelho = início do trecho provisório)")
ax[0].set_ylabel("pH")
df["y"].hist(bins=60, ax=ax[1])
ax[1].set_title("Distribuição do pH")
df.assign(hora=df["ds"].dt.hour).boxplot(column="y", by="hora", ax=ax[2], grid=False)
ax[2].set_title("pH por hora do dia (ciclo diário?)")
ax[2].set_xlabel("hora")
fig.tight_layout()
fig.savefig(OUT / "figs" / "01-eda.png")
print("fig salva:", OUT / "figs" / "01-eda.png")


maior gap: 18 passos = 1.5 h | gaps > 24 passos: 0


fig salva: /home/marcos/Projetos/temporal-model/resultados/02-lstnet-ph/figs/01-eda.png


## 3. Limpeza — grade completa + interpolação limitada
Reindex na grade de 5 min, flag do trecho provisório e interpolação temporal de no máximo 2 h (igual ao 00).

In [4]:
idx = pd.date_range(df["ds"].min(), df["ds"].max(), freq="5min")
s_raw = df.set_index("ds")["y"].reindex(idx)
print(f"slots na grade: {len(s_raw)} | linhas no CSV: {len(df)}")
s = s_raw.interpolate(method="time", limit=INTERP_LIMIT)
print(f"NaN após interpolação (limite {INTERP_LIMIT}): {int(s.isna().sum())}")
provisorio = s.index > pd.Timestamp(PROVISORIO_CORTE)
print(f"trecho provisório: {int(provisorio.sum())} slots ({100*provisorio.mean():.1f}%)")

amostra = slice("2026-06-08", "2026-06-15")
fig, ax = plt.subplots(figsize=(12, 3))
ax.plot(s_raw[amostra].index, s_raw[amostra].values, ".", ms=2, label="cru (com faltantes)")
ax.plot(s[amostra].index, s[amostra].values, lw=0.8, label=f"interpolado (limite {INTERP_LIMIT})")
ax.legend(); ax.set_title("Exemplo de preenchimento — semana 08–15/06")
fig.tight_layout(); fig.savefig(OUT / "figs" / "02-limpeza.png")
print("fig salva")


slots na grade: 26209 | linhas no CSV: 26209
NaN após interpolação (limite 24): 0
trecho provisório: 2484 slots (9.5%)


fig salva


## 4. Estacionariedade (ADF) e decomposição STL
Idêntico ao 00 (últimos 4032 pontos do treino, período 288).

In [5]:
n_total = len(s)
n_train = int(n_total * 0.70)
train = s.iloc[:n_train].dropna()
stat, pval, *_ = adfuller(train.values)
print(f"ADF stat={stat:.2f} p-valor={pval:.3g} → {'estacionária' if pval < 0.05 else 'NÃO estacionária'}")

stl = STL(train.iloc[-4032:], period=SEASON, robust=True).fit()
fig = stl.plot()
fig.set_size_inches(12, 6)
fig.savefig(OUT / "figs" / "03-stl.png")
print("fig salva")


ADF stat=-4.66 p-valor=0.000102 → estacionária


fig salva


## 5. Janelamento + holdout puro
Amostras `(L=8640 → H=288)` por janela deslizante, só janelas 100% observadas. Pré-holdout: split 70/15/15 **sem shuffle**. Holdout: últimos 10 dias + 10 origens diárias. **Idêntico ao 00** — o LSTM será avaliado nestas mesmas janelas.

In [6]:
from numpy.lib.stride_tricks import sliding_window_view

v = s.to_numpy()
W = sliding_window_view(v, L + H)
ok = ~np.isnan(W).any(axis=1)
W = W[ok]
X, Y = W[:, :L], W[:, L:]
ends = s.index[L + H - 1:][ok]
n = len(X)
ZONE = s.index.max() - pd.Timedelta(days=HOLDOUT_DIAS)
is_hold = ends >= (ZONE + pd.Timedelta(minutes=5 * (H - 1)))
ho = np.where(is_hold)[0]
pre = np.where(~is_hold)[0]
i1, i2 = int(len(pre) * 0.70), int(len(pre) * 0.85)
tr, va, te = pre[:i1], pre[i1:i2], pre[i2:]
splits = {"train": tr, "val": va, "test": te, "holdout": ho}
for k, idx in splits.items():
    print(f"{k}: {len(idx)} janelas | alvos {ends[idx[0]].date()} → {ends[idx[-1]].date()}")
print(f"janelas descartadas (com NaN): {len(s) - L - H + 1 - n}")
print(f"zona holdout (alvos): {ZONE.date()} → {s.index.max().date()}")
daily_ends = [ZONE + pd.Timedelta(minutes=5 * (H - 1 + H * k)) for k in range(HOLDOUT_DIAS)]
daily_idx = np.array([int(np.where(ends == d)[0][0]) for d in daily_ends])
print("dias previstos:", [str(ends[i].date()) for i in daily_idx])
TR_END = ends[tr[-1]]


train: 10281 janelas | alvos 2026-07-01 → 2026-08-06
val: 2203 janelas | alvos 2026-08-06 → 2026-08-14
test: 2204 janelas | alvos 2026-08-14 → 2026-08-21
holdout: 2594 janelas | alvos 2026-08-21 → 2026-08-31
janelas descartadas (com NaN): 0
zona holdout (alvos): 2026-08-21 → 2026-08-31
dias previstos: ['2026-08-21', '2026-08-22', '2026-08-23', '2026-08-24', '2026-08-25', '2026-08-26', '2026-08-27', '2026-08-28', '2026-08-29', '2026-08-30']


## 6. Baselines baratos (teste rolante + holdout)
Persistência, sazonal-naive (lag 288) e média móvel 288 — vetorizados, mesmos do 00.

In [7]:
def mae(a, b): return float(mean_absolute_error(a.ravel(), b.ravel()))
def rmse(a, b): return float(np.sqrt(mean_squared_error(a.ravel(), b.ravel())))
def mape(a, b, eps=1e-6): return float(np.mean(np.abs((a - b) / np.maximum(np.abs(a), eps))) * 100)
def smape(a, b, eps=1e-6): return float(np.mean(2*np.abs(a-b) / (np.abs(a)+np.abs(b)+eps)) * 100)

def metricas(y_true, y_pred):
    return {"MAE": mae(y_true, y_pred), "RMSE": rmse(y_true, y_pred),
            "MAPE": mape(y_true, y_pred), "sMAPE": smape(y_true, y_pred)}

def cheap_preds(X_):
    return {
        "persistencia": np.repeat(X_[:, -1:], H, axis=1),
        "sazonal_naive_288": np.stack([X_[:, L - SEASON + h] for h in range(H)], axis=1),
        "media_movel_288": np.repeat(X_[:, -SEASON:].mean(axis=1, keepdims=True), H, axis=1),
    }

Xte, Yte = X[te], Y[te]
Xho, Yho = X[ho], Y[ho]
pred_te = cheap_preds(Xte)
pred_ho = cheap_preds(Xho)
print("teste rolante:")
print(pd.DataFrame({m: metricas(Yte, p) for m, p in pred_te.items()}).T.round(4).to_string())


teste rolante:
                      MAE    RMSE    MAPE   sMAPE
persistencia       0.0727  0.0942  1.1683  1.1657
sazonal_naive_288  0.0501  0.0648  0.8079  0.8051
media_movel_288    0.0631  0.0768  1.0159  1.0133


## 7. LSTNet nativo — treino
Conv1D (k=12, stride 6) sobre `LN=2016` passos nativos + canais hora-do-dia → GRU + recurrent-skip `p=48` (mesma fase do dia anterior) + **atalho AR-288 linear em paralelo** (escala/nível) + RevIN por janela (afim aprendida). Saída **direta H=288 em 5 min, sem expansão**. Subamostra do treino/val por stride 4 (custo) — **avaliação (§8–§9) usa todas as origens**. Early stopping na val (MSE nativa).

In [8]:
SIN5 = np.sin(2*np.pi*(s.index.hour.to_numpy()*60+s.index.minute.to_numpy())/1440.0).astype(np.float32)
COS5 = np.cos(2*np.pi*(s.index.hour.to_numpy()*60+s.index.minute.to_numpy())/1440.0).astype(np.float32)
val5 = s.to_numpy().astype(np.float32)
Wln = sliding_window_view(val5, LN)
Tln = sliding_window_view(np.stack([SIN5, COS5], axis=1), LN, axis=0).transpose(0, 2, 1).astype(np.float32)
pos_end = s.index.get_indexer(ends - pd.Timedelta(minutes=5*H))
rowln = pos_end - LN + 1
print(f"janelas nativas válidas: {int((rowln >= 0).sum())}/{len(ends)} | Wln {Wln.shape}")
(OUT / "modelos").mkdir(parents=True, exist_ok=True)
json.dump({"mode": "revin-per-window", "LN": LN, "HN": HN},
          open(OUT / "modelos" / "normalizacao.json", "w"))

def monta(idxs):
    ii = np.asarray(idxs); r = rowln[ii]
    return Wln[r], Tln[r], Y[ii].astype(np.float32)

Xtr_v, Xtr_t, Ytr = monta(tr[::TRAIN_STRIDE])
Xva_v, Xva_t, Yva = monta(va[::VAL_STRIDE])
print(f"treino: {Xtr_v.shape} (stride {TRAIN_STRIDE}) | val: {Xva_v.shape} (stride {VAL_STRIDE})")

class LSTNet1D(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Conv1d(3, CONV_CH, kernel_size=CONV_K, stride=CONV_S)
        self.gru = nn.GRU(CONV_CH, GRU_H, batch_first=True)
        self.skipcell = nn.GRUCell(CONV_CH, SKIP_H)
        self.head = nn.Linear(GRU_H + SKIP_H, HN)
        self.ar = nn.Linear(AR_Q, HN)
        self.drop = nn.Dropout(DROPOUT)
        self.gamma = nn.Parameter(torch.ones(1))
        self.beta = nn.Parameter(torch.zeros(1))
    def forward(self, xv, tod):
        mu = xv.mean(dim=1, keepdim=True); sg = xv.std(dim=1, keepdim=True).clamp_min(1e-3)
        vn = self.gamma * (xv - mu) / sg + self.beta
        f = self.drop(torch.relu(self.conv(torch.cat([vn.unsqueeze(1), tod.transpose(1, 2)], dim=1))))
        f = f.transpose(1, 2)
        _, h = self.gru(f)
        B, T, _ = f.shape
        hs = torch.zeros(B, SKIP_H, device=f.device)
        states = [hs]
        for t in range(T):
            prev = states[t - SKIP_P] if t - SKIP_P >= 0 else states[0]
            hs = self.skipcell(f[:, t, :], prev)
            states.append(hs)
        g = self.gamma.clamp_min(1e-3)
        yn = self.head(self.drop(torch.cat([h.squeeze(0), hs], dim=1)))
        ya = self.ar(vn[:, -AR_Q:])
        return (yn + ya - self.beta) / g * sg + mu

model = LSTNet1D().to(DEVICE)
opt = torch.optim.Adam(model.parameters(), lr=LR)
loss_fn = nn.MSELoss()
tr_loader = DataLoader(TensorDataset(torch.from_numpy(Xtr_v), torch.from_numpy(Xtr_t), torch.from_numpy(Ytr)),
                       batch_size=BATCH, shuffle=True)
va_loader = DataLoader(TensorDataset(torch.from_numpy(Xva_v), torch.from_numpy(Xva_t), torch.from_numpy(Yva)),
                       batch_size=256)
n_params = sum(p.numel() for p in model.parameters())
print(f"params: {n_params}")

best, patience, hist = float("inf"), 0, {"train": [], "val": []}
t0 = time.time()
for ep in range(1, MAX_EPOCHS + 1):
    model.train()
    tl = 0.0
    for xb, tb, yb in tr_loader:
        opt.zero_grad()
        loss = loss_fn(model(xb, tb), yb)
        loss.backward()
        opt.step()
        tl += float(loss.detach()) * len(xb)
    tl /= len(tr_loader.dataset)
    model.eval()
    vl = 0.0
    with torch.no_grad():
        for xb, tb, yb in va_loader:
            vl += float(loss_fn(model(xb, tb), yb)) * len(xb)
    vl /= len(va_loader.dataset)
    hist["train"].append(tl); hist["val"].append(vl)
    tag = ""
    if vl < best:
        best, patience = vl, 0
        torch.save({"state": model.state_dict(),
                    "cfg": {"ln": LN, "conv": [CONV_CH, CONV_K, CONV_S],
                            "gru": GRU_H, "skip": [SKIP_H, SKIP_P], "ar": AR_Q}},
                   OUT / "modelos" / "lstnet_ph.pt")
        tag = " *"
    else:
        patience += 1
    print(f"ep {ep:02d} train={tl:.4f} val={vl:.4f}{tag}", flush=True)
    if patience >= PATIENCE:
        print(f"early stopping na ep {ep} (best val={best:.4f})")
        break
print(f"treino em {time.time()-t0:.0f}s | melhor val={best:.4f} | modelo: modelos/lstnet_ph.pt")

ckpt = torch.load(OUT / "modelos" / "lstnet_ph.pt", map_location="cpu", weights_only=False)
model.load_state_dict(ckpt["state"])
model.eval()

fig, ax = plt.subplots(figsize=(8, 3.5))
ax.plot(hist["train"], label="treino")
ax.plot(hist["val"], label="val")
ax.set_title("LSTNet — loss por época (MSE nativa 5 min)")
ax.set_xlabel("época"); ax.legend()
fig.tight_layout(); fig.savefig(OUT / "figs" / "07-curvas-treino.png")
print("fig salva: 07-curvas-treino.png")


janelas nativas válidas: 17282/17282 | Wln (24194, 2016)
treino: (2571, 2016) (stride 4) | val: (551, 2016) (stride 4)


params: 137506


ep 01 train=0.0151 val=0.0052 *


ep 02 train=0.0081 val=0.0053


ep 03 train=0.0070 val=0.0038 *


...

ep 26 train=0.0043 val=0.0036


ep 27 train=0.0043 val=0.0036


ep 28 train=0.0042 val=0.0034


early stopping na ep 28 (best val=0.0031)
treino em 619s | melhor val=0.0031 | modelo: modelos/lstnet_ph.pt


fig salva: 07-curvas-treino.png


## 8. Avaliação do LSTNet nas janelas do protocolo
Inferência **direta em 5 min** em **todas** as origens do teste rolante, do holdout e do holdout diário — sem agregação, sem expansão. Comparação direta com o alvo `Y` nativo.

In [9]:
@torch.no_grad()
def prevê(idxs, batch=256):
    ii = np.asarray(idxs)
    outs = []
    for b in range(0, len(ii), batch):
        xb = torch.from_numpy(Wln[rowln[ii[b:b+batch]]])
        tb = torch.from_numpy(Tln[rowln[ii[b:b+batch]]])
        outs.append(model(xb, tb).numpy())
    return np.concatenate(outs)

t0 = time.time()
Pn_te, Pn_ho, Pn_d = prevê(te), prevê(ho), prevê(daily_idx)
print(f"inferência em {time.time()-t0:.0f}s | teste {Pn_te.shape} holdout {Pn_ho.shape} diário {Pn_d.shape}")
print("LSTNet teste rolante:", {k: round(v, 4) for k, v in metricas(Yte, Pn_te).items()})
print("LSTNet holdout diário:", {k: round(v, 4) for k, v in metricas(Y[daily_idx], Pn_d).items()})


inferência em 8s | teste (2204, 288) holdout (2594, 288) diário (10, 288)
LSTNet teste rolante: {'MAE': 0.0456, 'RMSE': 0.0605, 'MAPE': 0.7351, 'sMAPE': 0.7318}
LSTNet holdout diário: {'MAE': 0.0446, 'RMSE': 0.0617, 'MAPE': 0.7209, 'sMAPE': 0.7173}


## 9. Comparação final + holdout dia a dia
Tabela do teste rolante (todas as origens), tabela do holdout diário (10 dias) e MAE por dia. Réguas do 00 impressas para referência.

In [10]:
linhas = {m: metricas(Yte, p) for m, p in pred_te.items()}
linhas["lstnet"] = metricas(Yte, Pn_te)
tab = pd.DataFrame(linhas).T.round(4)
tab.to_csv(OUT / "metricas_baseline.csv")
print("=== teste rolante ===")
print(tab.to_string())

Yd = Y[daily_idx]
diario = {m: metricas(Yd, cheap_preds(X[daily_idx])[m]) for m in pred_te}
diario["lstnet"] = metricas(Yd, Pn_d)
tab_d = pd.DataFrame(diario).T.round(4)
tab_d.to_csv(OUT / "metricas_holdout.csv")
print("=== holdout diário (10 dias) ===")
print(tab_d.to_string())

por_dia = pd.DataFrame(
    {m: [mae(Yd[k:k+1], cheap_preds(X[daily_idx])[m][k:k+1]) for k in range(len(Yd))]
     for m in pred_te},
    index=[str(ends[i].date()) for i in daily_idx])
por_dia["lstnet"] = [mae(Yd[k:k+1], Pn_d[k:k+1]) for k in range(len(Yd))]
print(por_dia.round(4).to_string())
print(f"\nRégua 00 (teste rolante): sazonal_naive_288 = 0.0501 | este exp: {tab['MAE'].idxmin()} = {tab['MAE'].min():.4f}")
print(f"Régua 00 (holdout diário): sazonal_naive_288 = 0.0466 | este exp: {tab_d['MAE'].idxmin()} = {tab_d['MAE'].min():.4f}")


=== teste rolante ===
                      MAE    RMSE    MAPE   sMAPE
persistencia       0.0727  0.0942  1.1683  1.1657
sazonal_naive_288  0.0501  0.0648  0.8079  0.8051
media_movel_288    0.0631  0.0768  1.0159  1.0133
lstnet             0.0456  0.0605  0.7351  0.7318
=== holdout diário (10 dias) ===
                      MAE    RMSE    MAPE   sMAPE
persistencia       0.0973  0.1204  1.5774  1.5599
sazonal_naive_288  0.0466  0.0673  0.7501  0.7497
media_movel_288    0.0655  0.0823  1.0547  1.0540
lstnet             0.0446  0.0617  0.7209  0.7173


            persistencia  sazonal_naive_288  media_movel_288  lstnet
2026-08-21        0.0642             0.0317           0.0562  0.0308
2026-08-22        0.0686             0.0363           0.0485  0.0323
2026-08-23        0.0725             0.0444           0.0631  0.0365
2026-08-24        0.0972             0.0414           0.0508  0.0445
2026-08-25        0.0589             0.0380           0.0498  0.0287
2026-08-26        0.1066             0.0370           0.0492  0.0484
2026-08-27        0.0781             0.0677           0.0792  0.0473
2026-08-28        0.1153             0.0523           0.0909  0.0523
2026-08-29        0.1642             0.0537           0.0981  0.0615
2026-08-30        0.1478             0.0633           0.0695  0.0640

Régua 00 (teste rolante): sazonal_naive_288 = 0.0501 | este exp: lstnet = 0.0456
Régua 00 (holdout diário): sazonal_naive_288 = 0.0466 | este exp: lstnet = 0.0446


In [11]:
E = ends[te]
fig, axes = plt.subplots(3, 1, figsize=(12, 9))
for ax, k in zip(axes, [0, len(Xte)//2, -1]):
    tc = pd.date_range(E[k] - pd.Timedelta(minutes=5*(H+2015)), E[k] - pd.Timedelta(minutes=5*H), freq="5min")
    ax.plot(tc, Xte[k][-2016:], lw=0.8, label="contexto (cauda 7d)")
    tf = pd.date_range(E[k] - pd.Timedelta(minutes=5*(H-1)), E[k], freq="5min")
    ax.plot(tf, Yte[k], "k-", lw=1.5, label="real")
    ax.plot(tf, pred_te["sazonal_naive_288"][k], "--", lw=1, label="sazonal-naive")
    ax.plot(tf, pred_te["persistencia"][k], ":", lw=1, label="persistência")
    ax.plot(tf, Pn_te[k], lw=1, alpha=0.9, label="lstnet (×12)")
    ax.set_title(f"origem {E[k]}")
    ax.legend(fontsize=8)
fig.tight_layout(); fig.savefig(OUT / "figs" / "04-forecasts.png")

fig, ax = plt.subplots(figsize=(8, 4))
tab["MAE"].sort_values().plot.barh(ax=ax)
ax.set_title("MAE no teste rolante — baselines + LSTNet (menor = melhor)")
fig.tight_layout(); fig.savefig(OUT / "figs" / "05-mae.png")

fig, axes = plt.subplots(5, 2, figsize=(14, 12), sharey=False)
for ax, k in zip(axes.ravel(), range(len(Yd))):
    tf = pd.date_range(ends[daily_idx[k]] - pd.Timedelta(minutes=5*(H-1)), ends[daily_idx[k]], freq="5min")
    ax.plot(tf, Yd[k], "k-", lw=1.2, label="real")
    ax.plot(tf, cheap_preds(X[daily_idx])["persistencia"][k], ":", lw=1, label="persistência")
    ax.plot(tf, cheap_preds(X[daily_idx])["sazonal_naive_288"][k], "--", lw=1, label="sazonal-naive")
    ax.plot(tf, Pn_d[k], lw=1, alpha=0.9, label="lstnet")
    ax.set_title(f"dia previsto {ends[daily_idx[k]].date()} (MAE lstnet={por_dia['lstnet'].iloc[k]:.3f} vs saz={por_dia['sazonal_naive_288'].iloc[k]:.3f})")
    ax.legend(fontsize=7)
fig.tight_layout(); fig.savefig(OUT / "figs" / "06-holdout-dias.png")
print("figs salvas")


figs salvas


## 10. Conclusões e próximos passos

- A régua do 00 (sazonal-naive 0,0501 / 0,0466) está impressa na §9 para comparação direta.
- O LSTNet opera nativo em 5 min com viés sazonal explícito (skip p=48 = ontem-mesma-hora) e âncora de escala (AR-288); julgue-o pelo MAE nas mesmas janelas do 00/01.
- Se empatar ou perder: candidatos seguintes são (i) PatchTST nativo (§3.3 do README), (ii) DLinear-5min/LightGBM com lags (§3.2), (iii) ablação do atalho AR (isolar sua contribuição).
- Artefatos em `resultados/02-lstnet-ph/`: `metricas_baseline.csv`, `metricas_holdout.csv`, `modelos/lstnet_ph.pt`, `modelos/normalizacao.json` e `figs/` (inclui `07-curvas-treino.png`).